# Project: Insurance Claims Intake Agent
**Brief from Paras:**
Watches Gmail for claims, extracts policy/date/value from email + PDF,
looks up policyholder in Notion/Airtable, creates record, auto-approves below $5k or escalates.
Includes Presidio PII redaction as the first node.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class InsuranceClaimsState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    raw_email_id: str
    raw_email_body: str
    raw_email_body_redacted: Optional[str]
    policy_number: Optional[str]
    incident_date: Optional[str]
    claim_value: Optional[float]
    policyholder_found: bool
    decision: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# redactor - Presidio PII redactor (pure Python, no Composio)
# No Composio actions; pure LLM or custom logic.
# TODO: implement redactor_node(state) following the pattern in 02_support_resolver.
def redactor_node(state):
    """Run Presidio analyzer + anonymizer on raw_email_body. Redact PERSON, US_SSN, PHONE, EMAIL, LOCATION."""
    raise NotImplementedError('TODO: implement redactor_node')


# extractor - Claim field extractor (LLM only)
# No Composio actions; pure LLM or custom logic.
# TODO: implement extractor_node(state) following the pattern in 02_support_resolver.
def extractor_node(state):
    """Extract policy_number, incident_date, claim_value from redacted text. Return JSON."""
    raise NotImplementedError('TODO: implement extractor_node')


# policy_lookup - Notion policy DB lookup
policy_lookup_tools = toolset.get_tools(actions=[Action.NOTION_QUERY_DATABASE])
policy_lookup_agent = create_react_agent(llm, policy_lookup_tools, prompt='''Look up policy_number in Notion 'Policies' DB. Return policyholder_found and policy_state.''')
def policy_lookup_node(state):
    result = policy_lookup_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='policy_lookup')]}


# router - Threshold router (custom @tool)
# No Composio actions; pure LLM or custom logic.
# TODO: implement router_node(state) following the pattern in 02_support_resolver.
def router_node(state):
    """Apply auto-approval rules. Use evaluate_claim_threshold custom tool."""
    raise NotImplementedError('TODO: implement router_node')


# notifier - Slack notifier
notifier_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
notifier_agent = create_react_agent(llm, notifier_tools, prompt='''Post decision to #claims-ops channel with audit trail.''')
def notifier_node(state):
    result = notifier_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='notifier')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('raw_email_body_redacted') is None: return {'next_worker': 'redactor'}
    if state.get('policy_number') is None: return {'next_worker': 'extractor'}
    if not state['policyholder_found']: return {'next_worker': 'policy_lookup'}
    if state.get('decision') is None: return {'next_worker': 'router'}
    if 'posted' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'notifier'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'router', 'policy_lookup', 'redactor', 'notifier', 'extractor'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("redactor", redactor_node)
g.add_node("extractor", extractor_node)
g.add_node("policy_lookup", policy_lookup_node)
g.add_node("router", router_node)
g.add_node("notifier", notifier_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "redactor": "redactor",
    "extractor": "extractor",
    "policy_lookup": "policy_lookup",
    "router": "router",
    "notifier": "notifier",
    "__end__": END,
})
g.add_edge("redactor", "supervisor")
g.add_edge("extractor", "supervisor")
g.add_edge("policy_lookup", "supervisor")
g.add_edge("router", "supervisor")
g.add_edge("notifier", "supervisor")

conn = sqlite3.connect("06_insurance_claims.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '06_insurance_claims-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'raw_email_id': '1001',
        'raw_email_body': 'My name is John Smith, policy A-12345, loss on 2026-04-01, claim is $3,200 for water damage.',
        'messages': [HumanMessage(content='New claim arrived')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
